# Day 4 homework — agents and tools (project)

**Course** Day 4 centers on the **DataTalks FAQ** (`data-engineering` filter). Your homework is the same pattern on **documentation you indexed** — here: **`pydantic/pydantic`** chunks + **minsearch** — then a **Pydantic AI** agent with a **`doc_search`** tool.

```bash
uv add openai python-dotenv pydantic-ai minsearch requests python-frontmatter
uv add --dev jupyter
```

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY (e.g. ai_hero/.env).")

from openai import OpenAI

openai_client = OpenAI()

In [6]:
import io
import zipfile

import frontmatter
import requests
from minsearch import Index


def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"skip {fn}: {e}")
    zf.close()
    return out


def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= n:
            break
    return result


pydantic_docs = read_repo_data("pydantic", "pydantic")
pydantic_chunks = []
for doc in pydantic_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    for ch in sliding_window(doc_content, 2000, 1000):
        ch.update(doc_copy)
        pydantic_chunks.append(ch)

MAX_CHUNKS = 400  # set None to index every chunk (slower)
_rows = pydantic_chunks if MAX_CHUNKS is None else pydantic_chunks[:MAX_CHUNKS]

doc_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[],
)
doc_index.fit(_rows)

len(pydantic_docs), len(_rows)

(98, 400)

## Pydantic AI agent + `doc_search` tool

Use **`run_agent_sync`** below (runs `asyncio.run` in a **thread**) so you avoid *event loop is already running* under Jupyter. Alternatively: **`result = await agent.run("...")`** if your kernel supports top-level await.

In [7]:
import asyncio
import threading
from typing import Any, List

from pydantic_ai import Agent


def run_agent_sync(agent, user_prompt: str):
    """Run agent.run in a thread with asyncio.run (works when Jupyter has a running loop)."""
    out: dict = {}

    def _go():
        out["r"] = asyncio.run(agent.run(user_prompt))

    t = threading.Thread(target=_go)
    t.start()
    t.join()
    return out["r"]


def doc_search(query: str) -> List[Any]:
    """
    Search the indexed Pydantic documentation chunks (markdown from the GitHub repo).

    Args:
        query: Keywords or a short question to look up.

    Returns:
        Up to 5 matching chunks with metadata.
    """
    return doc_index.search(query, num_results=5)


instructions = """
You help users with questions about the Pydantic Python library.
Always call `doc_search` first to ground answers in the official documentation chunks.
If search results are weak, say so and suggest what to try next.
""".strip()

agent = Agent(
    "openai:gpt-4o-mini",
    name="pydantic_docs_agent",
    instructions=instructions,
    tools=[doc_search],
)

result = run_agent_sync(agent, "How do I define a model with optional fields?")
print(result.output)

RuntimeError: This event loop is already running

In [ ]:
result.new_messages()

## Homework

- Point the pipeline at **your** repo/chunks from Days 1–3.  
- Tune **instructions** and compare **lexical** tool vs (optional) **vector** retrieval wired into the same tool name.  
- Post about the agent.

[Course](https://alexeygrigorev.com/aihero/) · [Slack `#course-ai-hero`](https://datatalks.club/)